<a href="https://colab.research.google.com/github/Miguel3o2/flyrank-demo/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Setup — connect DuckDB to the hosted warehouse

Same pattern as the Week-3+ notebook: token via env var → Colab Secret → `getpass` prompt
(never pasted into a cell), then `TABLES` pointing at the hosted Parquet.

If you don't have a `HF_TOKEN` handy right now, skip to the **local-CSV fallback** cell further
down instead — everything after Section 0 works off the `sig` dataframe regardless of which
path built it.


In [1]:
%pip -q install duckdb huggingface_hub


In [2]:
import os, getpass, json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...), or leave blank to use the local CSV fallback below: ")


Paste your Hugging Face READ token (hf_...), or leave blank to use the local CSV fallback below: ··········


In [3]:
import duckdb

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent.parent
elif REPO_ROOT.name == "work":
    REPO_ROOT = REPO_ROOT.parent
OUT_DIR = REPO_ROOT / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
USING_WAREHOUSE = bool(HF_TOKEN)

if USING_WAREHOUSE:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

    REL = "hf://datasets/FlyRank/internship-warehouse"
    TABLES = {
        "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
        "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
        "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
        "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
        "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
    }
    for name, src in TABLES.items():
        n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
        print(f"{name:22} {n:>12,} rows")
else:
    print("No HF_TOKEN provided — will use the local-CSV fallback cell below instead.")


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


### Know your panel before you score it

Client history depth differs (unbalanced panel) — `dim_clients` says exactly what each client
has. Worth a glance before trusting any window across the whole warehouse.


In [4]:
if USING_WAREHOUSE:
    clients = con.sql(f"""
        SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
        FROM {TABLES['dim_clients']}
        ORDER BY gsc_data_start NULLS LAST
    """).df()
    display(clients.head(10))


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


In [5]:
if USING_WAREHOUSE:
    print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df())


                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

### Local-CSV fallback (only run this if `USING_WAREHOUSE` is `False`)

Uses the small starter CSV instead (`content_refresh_anonymized.csv` /
`refresh_feature_vector.csv`) if you don't have a Hugging Face token handy. Auto-detects
columns so it survives minor schema differences.


In [6]:
if not USING_WAREHOUSE:
    CANDIDATE_PATHS = [
        REPO_ROOT / "data" / "processed" / "refresh_feature_vector.csv",
        REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv",
    ]
    csv_path = next((p for p in CANDIDATE_PATHS if p.exists()), None)
    if csv_path is None:
        raise FileNotFoundError(f"No HF_TOKEN and no local CSV found. Checked: {CANDIDATE_PATHS}")
    fallback_df = pd.read_csv(csv_path)
    print(f"Loaded {csv_path} -> {fallback_df.shape}")
    print(list(fallback_df.columns))


## Build the current-state feature table (SQL does the work, not RAM)

Same shape as Notebook 3's `features` / `qsignals` cells: aggregate the last complete 30-day
window per `(client_hash_id, content_hash_id)`, plus the 90-day query-mix signals. Nothing
here reaches past "today" — `last30` is the most recent complete window in the warehouse, used
as a snapshot of current state, not a held-out future period.

This is the heaviest cell — expect a couple of minutes on Colab. If it stalls or 429s, swap
`TABLES['fact_daily']` for `TABLES['fact_daily_sample']` and re-run.


In [7]:
if USING_WAREHOUSE:
    features = con.sql(f"""
        WITH bounds AS (
            SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
        ),
        windowed AS (
            SELECT f.client_hash_id, f.content_hash_id,
                   SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
                   SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
                   SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
                   AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30,
                   STDDEV(f.gsc_avg_position)                                                                   AS pos_stddev_90d
            FROM {TABLES['fact_daily']} f, bounds b
            WHERE f.report_date > b.end_d - INTERVAL 90 DAY
            GROUP BY 1, 2
            HAVING imp_last30 >= 200   -- need enough last-30d impressions to trust a CTR
        )
        SELECT * FROM windowed
    """).df()

    print(f"{len(features):,} content items with enough current-window history")
    features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

80,861 content items with enough current-window history


In [8]:
if USING_WAREHOUSE:
    qsignals = con.sql(f"""
        SELECT content_hash_id,
               ANY_VALUE(content_visible_query_count)  AS visible_queries,
               ANY_VALUE(rare_impressions_share)        AS rare_share,
               ANY_VALUE(anonymized_impressions_share)  AS anon_share,
               MAX(impressions_90d)                     AS top_query_impressions,
               SUM(impressions_90d)                      AS kept_impressions
        FROM {TABLES['fact_query_90d']}
        GROUP BY content_hash_id
    """).df()
    qsignals["top_query_share"] = qsignals["top_query_impressions"] / qsignals["kept_impressions"]

    sig = features.merge(qsignals, on="content_hash_id", how="left")
    sig["ctr_last30"] = np.where(sig["imp_last30"] > 0, sig["clk_last30"] / sig["imp_last30"], np.nan)
    print(f"joined: {len(sig):,} rows")
else:
    def find_col(frame, *substrings):
        cols_lower = {c: c.lower() for c in frame.columns}
        for sub in substrings:
            for c, cl in cols_lower.items():
                if sub in cl:
                    return c
        return None

    c_pos    = find_col(fallback_df, "avg_position", "position", "rank")
    c_ctr    = find_col(fallback_df, "ctr")
    c_clicks = find_col(fallback_df, "clicks")
    c_impr   = find_col(fallback_df, "impressions")
    c_vol    = find_col(fallback_df, "search_volume", "query_volume", "volume")
    c_content = find_col(fallback_df, "content_hash_id", "page_url", "url")

    sig = fallback_df.copy()
    sig["content_hash_id"] = sig[c_content] if c_content else sig.index.astype(str)
    sig["pos_last30"] = sig[c_pos] if c_pos else np.nan
    sig["ctr_last30"] = sig[c_ctr] if c_ctr else (
        sig[c_clicks] / sig[c_impr] if c_clicks and c_impr else np.nan
    )
    sig["visible_queries"] = sig[c_vol] if c_vol else np.nan
    sig["kept_impressions"] = sig[c_impr] if c_impr else np.nan
    sig["top_query_share"] = np.nan
    sig["pos_stddev_90d"] = np.nan
    print("Fallback signal table built:", sig.shape)

sig.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 80,861 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,pos_stddev_90d,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share,ctr_last30
0,client_e547b89c05043229,content_2e296120acb03e93,2346.0,5207.0,0.0,38.156281,9.232266,43.0,0.042632,0.545876,409.0,3108.0,0.131596,0.000000
1,client_e547b89c05043229,content_516b7c0e8eec0cef,371.0,2494.0,0.0,48.364954,13.601872,13.0,0.112391,0.679232,305.0,597.0,0.510888,0.000000
2,client_e547b89c05043229,content_38b6c1a9aa29f801,5746.0,12657.0,2.0,40.082120,9.400494,129.0,0.069119,0.517198,1398.0,7613.0,0.183633,0.000348
3,client_e547b89c05043229,content_2ffd36f2a70be7e3,1051.0,3738.0,0.0,27.835101,7.642705,30.0,0.061182,0.749635,145.0,906.0,0.160044,0.000000
4,client_e547b89c05043229,content_4724385fe790d24a,4771.0,2878.0,11.0,8.842269,3.586000,18.0,0.048242,0.868087,243.0,640.0,0.379688,0.002306


## 1. Signal checks (before the rule leans on them)

Two signals, one bucket table each, `n` printed, one-word verdict — both traced back to a real
FlyRank flag:

- **CTR-vs-position gap** → the CTR-fix logic
- **Query/impression volume** → the quick-win logic

The check: bucket the signal, then look at the rate of an observable-today "underperforming
despite the opportunity" pattern within each bucket — never a forward-looking outcome.


In [9]:


work = sig.dropna(subset=["pos_last30", "ctr_last30"]).copy()

work["_pos_bucket"] = pd.qcut(work["pos_last30"].rank(method="first"), 5, labels=False)
expected_ctr = work.groupby("_pos_bucket")["ctr_last30"].transform("median")
work["_ctr_gap"] = work["ctr_last30"] - expected_ctr  # negative = underperforming for its rank

work["_gap_bucket"] = pd.cut(
    work["_ctr_gap"],
    bins=[-np.inf, -0.03, -0.01, 0.01, 0.03, np.inf],
    labels=["<-3pp", "-3..-1pp", "-1..+1pp", "+1..+3pp", ">+3pp"],
)

if "clk_last30" in work.columns and "imp_last30" in work.columns:
    high_impr = work["imp_last30"] >= work["imp_last30"].median()
    low_clicks = work["clk_last30"] < work["clk_last30"].median()
    work["_ctr_fix_pattern"] = (high_impr & low_clicks).astype(int)
else:
    work["_ctr_fix_pattern"] = (work["_ctr_gap"] < 0).astype(int)

signal1_table = (
    work.groupby("_gap_bucket", observed=True)
    .agg(n=("_ctr_fix_pattern", "size"), pattern_rate=("_ctr_fix_pattern", "mean"))
    .round(3)
)
print("Signal 1 — CTR-vs-position gap bucket table (n and CTR-fix pattern rate)")
print(signal1_table)
print("Total n:", signal1_table["n"].sum())


Signal 1 — CTR-vs-position gap bucket table (n and CTR-fix pattern rate)
                 n  pattern_rate
_gap_bucket                     
-1..+1pp     76682           0.1
+1..+3pp      3924           0.0
>+3pp          255           0.0
Total n: 80861


```
SIGNAL 1 VERDICT: <fill in after reading the table — CONFIRMED / OPPOSITE / MIXED / FALSE>
Why: <one sentence, pointing at the actual numbers in signal1_table>
```


In [10]:


vol_basis = "kept_impressions" if sig["kept_impressions"].notna().any() else "visible_queries"
work2 = sig.dropna(subset=[vol_basis]).copy()

work2["_vol_bucket"] = pd.qcut(work2[vol_basis].rank(method="first"), 5, labels=False,
                                 duplicates="drop")

if "clk_last30" in work2.columns:
    low_clicks_overall = work2["clk_last30"] < work2["clk_last30"].median()
    work2["_quick_win_pattern"] = low_clicks_overall.astype(int)
else:
    work2["_quick_win_pattern"] = (work2["ctr_last30"] < work2["ctr_last30"].median()).astype(int)

signal2_table = (
    work2.groupby("_vol_bucket", observed=True)
    .agg(n=("_quick_win_pattern", "size"), pattern_rate=("_quick_win_pattern", "mean"))
    .round(3)
)
signal2_table.index = [f"vol_q{int(i)+1}" for i in signal2_table.index]
print(f"Signal 2 — volume ({vol_basis}) bucket table (n and quick-win pattern rate)")
print(signal2_table)
print("Total n:", signal2_table["n"].sum())


Signal 2 — volume (kept_impressions) bucket table (n and quick-win pattern rate)
            n  pattern_rate
vol_q1  15413         0.607
vol_q2  15413         0.589
vol_q3  15412         0.546
vol_q4  15413         0.398
vol_q5  15413         0.161
Total n: 77064


```
SIGNAL 2 VERDICT: <CONFIRMED / OPPOSITE / MIXED / FALSE>
Why: <one sentence tied to signal2_table>
```

A clearly-explained `FALSE` or `OPPOSITE` here is a win, not a failure — it just kept a bad
signal out of the rule below.


## 2. The rule — score, ONE reason code, an action label

Combine whichever signal(s) came back `CONFIRMED` above into one score. Edit `WEIGHTS` to
reflect the actual verdicts — don't score on something that just came back `FALSE`.


In [11]:
score_df = sig.copy()

pos_bucket_all = pd.qcut(score_df["pos_last30"].rank(method="first"), 5, labels=False, duplicates="drop")
expected_ctr_all = score_df.groupby(pos_bucket_all)["ctr_last30"].transform("median")
gap_all = (expected_ctr_all - score_df["ctr_last30"]).clip(lower=0)
score_df["ctr_gap_component"] = ((gap_all - gap_all.min()) / (gap_all.max() - gap_all.min() + 1e-9)).fillna(0)

vol_series = score_df[vol_basis]
score_df["volume_component"] = (
    (vol_series - vol_series.min()) / (vol_series.max() - vol_series.min() + 1e-9)
).clip(0, 1).fillna(0)

WEIGHTS = {
    "ctr_gap_component": 0.6,
    "volume_component": 0.4,
}
weight_sum = sum(WEIGHTS.values())
WEIGHTS = {k: v / weight_sum for k, v in WEIGHTS.items()}

score_df["baseline_score"] = sum(score_df[k] * w for k, w in WEIGHTS.items()).round(4)
print("Weights used:", WEIGHTS)
score_df[["baseline_score", "ctr_gap_component", "volume_component"]].describe()


Weights used: {'ctr_gap_component': 0.6, 'volume_component': 0.4}


,baseline_score,ctr_gap_component,volume_component
count,80861.000000,80861.000000,80861.000000
mean,0.118509,0.194737,0.004168
std,0.161315,0.267686,0.016545
min,0.000000,0.000000,0.000000
25%,0.000200,0.000000,0.000211
50%,0.010700,0.000000,0.000748
75%,0.201100,0.330108,0.002544
max,0.818100,1.000000,1.000000


In [12]:
def reason_code(row):
    contribs = {k: row[k] * WEIGHTS[k] for k in WEIGHTS}
    top = max(contribs, key=contribs.get)
    return {"ctr_gap_component": "CTR_UNDERPERFORM", "volume_component": "HIGH_VOLUME_OPPORTUNITY"}[top]

score_df["reason_code"] = score_df.apply(reason_code, axis=1)

def action_label(s):
    if s >= 0.66:
        return "REFRESH_NOW"
    elif s >= 0.33:
        return "MONITOR"
    return "NO_ACTION"

score_df["action"] = score_df["baseline_score"].apply(action_label)

print(score_df["reason_code"].value_counts())
print()
print(score_df["action"].value_counts())


reason_code
CTR_UNDERPERFORM           42841
HIGH_VOLUME_OPPORTUNITY    38020
Name: count, dtype: int64

action
NO_ACTION      68107
MONITOR        12752
REFRESH_NOW        2
Name: count, dtype: int64


In [13]:
id_cols = [c for c in ["client_hash_id", "content_hash_id"] if c in score_df.columns]
keep_cols = id_cols + [
    c for c in ["pos_last30", "ctr_last30", "imp_last30", "clk_last30", vol_basis, "pos_stddev_90d"]
    if c in score_df.columns
] + ["baseline_score", "reason_code", "action"]

queue = score_df[keep_cols].sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)

out_path = OUT_DIR / "baseline_action_score.csv"
queue.to_csv(out_path, index=False)
print(f"Wrote {len(queue):,} rows -> {out_path}")
queue.head(10)


Wrote 80,861 rows -> /content/work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,pos_last30,ctr_last30,imp_last30,clk_last30,kept_impressions,pos_stddev_90d,baseline_score,reason_code,action
0,1,client_8ddc46da5414ffd8,content_943dc881428182b8,2.689206,0.001364,292416.0,399.0,606884.0,1.675183,0.8181,CTR_UNDERPERFORM,REFRESH_NOW
1,2,client_8ddc46da5414ffd8,content_d0acf7062bc6b257,3.049897,0.000317,145022.0,46.0,297755.0,1.118224,0.7540,CTR_UNDERPERFORM,REFRESH_NOW
2,3,client_a80fca3f171ed1de,content_c1f764a2f362d1c3,7.132457,0.000191,235427.0,45.0,352652.0,1.083796,0.6514,CTR_UNDERPERFORM,MONITOR
3,4,client_8ddc46da5414ffd8,content_7471467133493ce6,2.793545,0.000700,75720.0,53.0,217559.0,0.896698,0.6501,CTR_UNDERPERFORM,MONITOR
4,5,client_23a62021009f63c4,content_bddfdd871aa09fbe,5.714560,0.000448,49074.0,22.0,158082.0,6.896983,0.6444,CTR_UNDERPERFORM,MONITOR
5,6,client_0fa64a184f18a4a0,content_11bf4c33adea7bdc,21.624960,0.000000,107215.0,0.0,332312.0,13.455677,0.6416,CTR_UNDERPERFORM,MONITOR
6,7,client_810019792c9b8efc,content_ceea051963ea5e73,3.293404,0.000000,1414.0,0.0,38169.0,3.022205,0.6252,CTR_UNDERPERFORM,MONITOR
7,8,client_62f4a7e64f5e0096,content_4537e097260c4a55,5.163721,0.000000,14057.0,0.0,29537.0,1.188819,0.6195,CTR_UNDERPERFORM,MONITOR
8,9,client_73cda7b4e4f265ea,content_b9acd1ebff7d34ff,3.477198,0.000000,1695.0,0.0,23677.0,1.878143,0.6156,CTR_UNDERPERFORM,MONITOR
9,10,client_a80fca3f171ed1de,content_acc98771fbf23545,6.076648,0.000161,18691.0,3.0,51553.0,0.710906,0.6126,CTR_UNDERPERFORM,MONITOR


## 3. Top-10 review

For each of the top ten: the action, why it's there (which reason code / signal drove it),
and what would make it wrong. Read the actual rows below before writing these — say something
concrete about that specific `content_hash_id`, not a paraphrase of the reason code.


In [14]:
top10 = queue.head(10)
top10


,rank,client_hash_id,content_hash_id,pos_last30,ctr_last30,imp_last30,clk_last30,kept_impressions,pos_stddev_90d,baseline_score,reason_code,action
0,1,client_8ddc46da5414ffd8,content_943dc881428182b8,2.689206,0.001364,292416.0,399.0,606884.0,1.675183,0.8181,CTR_UNDERPERFORM,REFRESH_NOW
1,2,client_8ddc46da5414ffd8,content_d0acf7062bc6b257,3.049897,0.000317,145022.0,46.0,297755.0,1.118224,0.7540,CTR_UNDERPERFORM,REFRESH_NOW
2,3,client_a80fca3f171ed1de,content_c1f764a2f362d1c3,7.132457,0.000191,235427.0,45.0,352652.0,1.083796,0.6514,CTR_UNDERPERFORM,MONITOR
3,4,client_8ddc46da5414ffd8,content_7471467133493ce6,2.793545,0.000700,75720.0,53.0,217559.0,0.896698,0.6501,CTR_UNDERPERFORM,MONITOR
4,5,client_23a62021009f63c4,content_bddfdd871aa09fbe,5.714560,0.000448,49074.0,22.0,158082.0,6.896983,0.6444,CTR_UNDERPERFORM,MONITOR
5,6,client_0fa64a184f18a4a0,content_11bf4c33adea7bdc,21.624960,0.000000,107215.0,0.0,332312.0,13.455677,0.6416,CTR_UNDERPERFORM,MONITOR
6,7,client_810019792c9b8efc,content_ceea051963ea5e73,3.293404,0.000000,1414.0,0.0,38169.0,3.022205,0.6252,CTR_UNDERPERFORM,MONITOR
7,8,client_62f4a7e64f5e0096,content_4537e097260c4a55,5.163721,0.000000,14057.0,0.0,29537.0,1.188819,0.6195,CTR_UNDERPERFORM,MONITOR
8,9,client_73cda7b4e4f265ea,content_b9acd1ebff7d34ff,3.477198,0.000000,1695.0,0.0,23677.0,1.878143,0.6156,CTR_UNDERPERFORM,MONITOR
9,10,client_a80fca3f171ed1de,content_acc98771fbf23545,6.076648,0.000161,18691.0,3.0,51553.0,0.710906,0.6126,CTR_UNDERPERFORM,MONITOR


1. **Row 1** — Action: `<...>`. Why: `<e.g. "CTR gap -4.1pp at position ~6, high last-30d
   impressions">`. What would make it wrong: `<e.g. "if this page is slated for consolidation
   into another URL, a refresh is wasted effort">`
2. **Row 2** — ...
3. **Row 3** — ...
4. **Row 4** — ...
5. **Row 5** — ...
6. **Row 6** — ...
7. **Row 7** — ...
8. **Row 8** — ...
9. **Row 9** — ...
10. **Row 10** — ...


## 4. Weak picks

Rows where the score leans almost entirely on one component — the rule has nothing else
corroborating the call. Say concretely why the rule could be fooled here.


In [15]:
component_cols = list(WEIGHTS.keys())
score_df["_max_component_share"] = score_df[component_cols].div(
    score_df[component_cols].sum(axis=1) + 1e-9, axis=0
).max(axis=1)

weak_candidates = (
    score_df[score_df["_max_component_share"] > 0.85]
    .sort_values("baseline_score", ascending=False)
    [keep_cols + ["_max_component_share"]]
    .head(5)
)
weak_candidates


,client_hash_id,content_hash_id,pos_last30,ctr_last30,imp_last30,clk_last30,kept_impressions,pos_stddev_90d,baseline_score,reason_code,action,_max_component_share
32392,client_810019792c9b8efc,content_ceea051963ea5e73,3.293404,0.000000,1414.0,0.0,38169.0,3.022205,0.6252,CTR_UNDERPERFORM,MONITOR,0.940842
58294,client_62f4a7e64f5e0096,content_4537e097260c4a55,5.163721,0.000000,14057.0,0.0,29537.0,1.188819,0.6195,CTR_UNDERPERFORM,MONITOR,0.953603
4630,client_73cda7b4e4f265ea,content_b9acd1ebff7d34ff,3.477198,0.000000,1695.0,0.0,23677.0,1.878143,0.6156,CTR_UNDERPERFORM,MONITOR,0.962466
26581,client_a80fca3f171ed1de,content_acc98771fbf23545,6.076648,0.000161,18691.0,3.0,51553.0,0.710906,0.6126,CTR_UNDERPERFORM,MONITOR,0.919056
30482,client_e00b29e582949543,content_b04adb0fcfd417f6,5.620795,0.000000,15866.0,0.0,18554.0,2.813453,0.6122,CTR_UNDERPERFORM,MONITOR,0.970349


1. **`<content_hash_id>`** — flagged `<action>` almost entirely on `<component>`. Concretely
   wrong if: `<...>`
2. **`<content_hash_id>`** — ...


## 5. Self-check

`is_declining`-style label logic is allowed **only here**, as a look-only sanity check —
mirroring how Notebook 3 defines its label — never as a rule input.


In [16]:
checks = {
    "csv_written": out_path.exists(),
    "csv_row_count": len(pd.read_csv(out_path)) if out_path.exists() else 0,
    "n_signal1": int(signal1_table["n"].sum()),
    "n_signal2": int(signal2_table["n"].sum()),
    "action_counts": queue["action"].value_counts().to_dict(),
    "reason_code_counts": queue["reason_code"].value_counts().to_dict(),
    "label_columns_excluded_from_rule": all(c not in WEIGHTS for c in ["is_declining"]),
}
print(json.dumps(checks, indent=2, default=str))


{
  "csv_written": true,
  "csv_row_count": 80861,
  "n_signal1": 80861,
  "n_signal2": 77064,
  "action_counts": {
    "NO_ACTION": 68107,
    "MONITOR": 12752,
    "REFRESH_NOW": 2
  },
  "reason_code_counts": {
    "CTR_UNDERPERFORM": 42841,
    "HIGH_VOLUME_OPPORTUNITY": 38020
  },
  "label_columns_excluded_from_rule": true
}


In [17]:
# Optional, look-only: same label shape as Notebook 3 (imp_last30 < 0.8 * imp_prev30) —
# does the rule's top action agree directionally? NOT used to build the score or weights.
if "imp_prev30" in score_df.columns and "imp_last30" in score_df.columns:
    peek = score_df.copy()
    peek["is_declining"] = (peek["imp_last30"] < 0.8 * peek["imp_prev30"]).astype(int)
    print(peek.groupby("action")["is_declining"].mean().round(3))
else:
    print("No prev30/last30 impressions pair available (local-CSV fallback) — nothing to peek at.")


action
MONITOR        0.750
NO_ACTION      0.732
REFRESH_NOW    0.500
Name: is_declining, dtype: float64


### Self-check summary (fill in by hand)

- [ ] Both signal checks show a printed bucket table with `n`.
- [ ] Both checked signals are tied to real FlyRank flags (CTR-vs-position → CTR-fix logic,
      volume → quick-win logic).
- [ ] Each signal has a one-word verdict with a one-sentence reason.
- [ ] The rule uses only signals that came back `CONFIRMED` (or explains why a `MIXED` one is
      still included).
- [ ] The rule produces exactly one score, one reason code per row, one action label per row.
- [ ] `work/outputs/baseline_action_score.csv` exists and regenerates by running top to bottom.
- [ ] Top-10 review has 10 rows, each with action / why / what-would-make-it-wrong.
- [ ] No future-window or label-derived column (`is_declining`) was used as a rule *input* —
      it only appears in this self-check section, as a look.

**Lane confirmation:** `<state your Week-5 lane here — the one this baseline needs to beat>`
